# Diamond Pipeline — Parse, Scrape, Enrich

1. **Step 1** — Parse ALL diamonds from `New Text Document.txt` → `diamonds_raw.csv`
2. **Step 2** — Visit each product page (15 threads) → get SKU → derive IGI PDF URL
3. **Step 3** — Download IGI PDFs (15 threads) → extract proportions with pdfplumber
4. **Step 4** — Merge old + new data → `diamonds_full.csv`

## 1. Install Dependencies

In [ ]:
!pip install -q "google-colab-selenium[undetected]" pdfplumber requests pandas beautifulsoup4

import os
os.environ["DISPLAY"] = ":99"
print("Dependencies installed.")

## 2. Imports & Config

In [ ]:
import re
import time
import io
import os
import json
import requests
import pdfplumber
import pandas as pd
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

from selenium.webdriver.common.by import By
from selenium.common.exceptions import TimeoutException, NoSuchElementException

# ── Config ─────────────────────────────────────────────────────
THREADS = 15               # Number of parallel threads for requests
HTML_FILE = "New Text Document.txt"
RAW_CSV = "diamonds_raw.csv"
FULL_CSV = "diamonds_full.csv"

print(f"Threads: {THREADS}")

## 3. Step 1 — Parse ALL Diamonds from HTML → `diamonds_raw.csv`

In [ ]:
def parse_all_diamonds(html_path):
    """
    Parse every <tr class='diamondList'> from the saved HTML.
    Returns a list of dicts with: shape, carat, color, clarity, cut,
    price_original, price_discounted, product_id.
    """
    with open(html_path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")

    rows = soup.select("tr.diamondList")
    print(f"Found {len(rows)} diamond rows in HTML")

    diamonds = []
    for row in rows:
        # Product ID from onclick
        onclick = row.get("onclick", "")
        pid_match = re.search(r'displayDetailView\(this,\s*(\d+)\)', onclick)
        product_id = pid_match.group(1) if pid_match else None

        cells = row.find_all("td")
        if len(cells) < 6:
            continue

        # Shape (col 0): "(R) Round"
        shape_text = cells[0].get_text(strip=True)
        shape_match = re.search(r'\(\w\)\s*(\w+)', shape_text)
        shape = shape_match.group(1) if shape_match else shape_text

        # Price (col 1): original in <del>, discounted is remaining text
        price_td = cells[1]
        price_original = None
        price_discounted = None

        del_tag = price_td.find("del")
        if del_tag:
            orig_match = re.search(r'\$([\d,]+)', del_tag.get_text())
            if orig_match:
                price_original = int(orig_match.group(1).replace(",", ""))
            del_tag.decompose()

        disc_match = re.search(r'\$([\d,]+)', price_td.get_text())
        if disc_match:
            price_discounted = int(disc_match.group(1).replace(",", ""))

        # Carat (col 2)
        carat_text = cells[2].get_text(strip=True)
        carat = float(carat_text) if re.match(r'^\d+\.?\d*$', carat_text) else None

        # Color (col 3), Clarity (col 4), Cut (col 5)
        color = cells[3].get_text(strip=True)
        clarity = cells[4].get_text(strip=True)
        cut = cells[5].get_text(strip=True)

        diamonds.append({
            "product_id": product_id,
            "shape": shape,
            "carat": carat,
            "color": color,
            "clarity": clarity,
            "cut": cut,
            "price_original": price_original,
            "price_discounted": price_discounted,
        })

    return diamonds


# ── Find and parse the HTML file ──────────────────────────────
html_path = None
for p in [f"/content/{HTML_FILE}", HTML_FILE, os.path.join(os.getcwd(), HTML_FILE)]:
    if os.path.exists(p):
        html_path = p
        break

if html_path is None:
    raise FileNotFoundError(
        f"'{HTML_FILE}' not found. Upload it to Colab or place it next to this notebook."
    )

all_diamonds = parse_all_diamonds(html_path)
print(f"\nParsed {len(all_diamonds)} diamonds total")

# ── Save to raw CSV ──────────────────────────────────────────
df_raw = pd.DataFrame(all_diamonds)
df_raw.to_csv(RAW_CSV, index=False)
print(f"Saved → {RAW_CSV}")

# ── Quick summary ────────────────────────────────────────────
print(f"\nShape breakdown:")
print(df_raw["shape"].value_counts().to_string())
print(f"\nCarat range: {df_raw['carat'].min()} – {df_raw['carat'].max()}")
print(f"Price range: ${df_raw['price_discounted'].min()} – ${df_raw['price_discounted'].max()}")
df_raw.head(10)

## 4. Grab Cloudflare Cookies with ONE Selenium Driver

Use a single Chrome instance to pass Cloudflare, then steal the cookies
for `requests.Session` — this lets us hit product pages from 15 threads
without needing 15 browsers.

In [ ]:
def create_driver():
    """Create a headless Chrome driver (Colab-compatible)."""
    try:
        import google_colab_selenium as gs
        driver = gs.UndetectedChrome()
        driver.implicitly_wait(8)
        return driver
    except Exception:
        pass
    try:
        import google_colab_selenium as gs
        driver = gs.Chrome()
        driver.implicitly_wait(8)
        return driver
    except Exception:
        pass

    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.chrome.service import Service

    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    for binary in ["/usr/bin/google-chrome-stable", "/usr/bin/google-chrome",
                   "/usr/bin/chromium-browser", "/usr/bin/chromium"]:
        if os.path.exists(binary):
            options.binary_location = binary
            break
    for drv in ["/usr/bin/chromedriver", "/usr/local/bin/chromedriver"]:
        if os.path.exists(drv):
            service = Service(drv)
            driver = webdriver.Chrome(service=service, options=options)
            driver.implicitly_wait(8)
            return driver
    raise RuntimeError("Could not create a Chrome driver")


def get_luvansh_session():
    """
    Use ONE Selenium driver to:
    1. Visit luvansh.com → pass Cloudflare challenge
    2. Visit one product page → confirm SKU is visible
    3. Extract all cookies + user-agent
    4. Quit the driver
    Returns a requests.Session pre-loaded with those cookies.
    """
    print("[Session] Starting Chrome to grab cookies ...")
    driver = create_driver()

    user_agent = driver.execute_script("return navigator.userAgent")
    print(f"[Session] User-Agent: {user_agent[:80]}...")

    try:
        # Step 1: Visit the shop page → triggers Cloudflare
        driver.get("https://www.luvansh.com/shop-diamond")
        time.sleep(5)

        # Wait for Cloudflare if needed
        for _ in range(3):
            if "just a moment" in driver.page_source.lower():
                print("[Session] Cloudflare challenge — waiting ...")
                time.sleep(5)
            else:
                break

        print(f"[Session] Page title: {driver.title}")

        # Step 2: Visit one product page to verify cookies work
        # Use first diamond from our parsed list
        test_d = all_diamonds[0]
        pid = test_d["product_id"]
        carat_str = str(test_d["carat"]).replace(".", "-")
        slug = (f"igi-certified-{carat_str}-{test_d['color'].lower()}-"
                f"{test_d['clarity'].lower()}-{test_d['shape'].lower()}-"
                f"lab-created-diamond-r")
        test_url = f"https://www.luvansh.com/product/{pid}/{slug}"

        print(f"[Session] Testing product page: {test_url}")
        driver.get(test_url)
        time.sleep(4)

        # Verify SKU is visible
        page_text = driver.find_element(By.TAG_NAME, "body").text
        sku_m = re.search(r'SKU\s*:\s*(LV-[A-Z0-9-]+)', page_text, re.I)
        if sku_m:
            print(f"[Session] Test SKU found: {sku_m.group(1)}")
        else:
            print(f"[Session] WARNING: SKU not found on test page")
            # Show what we see
            for line in page_text.split("\n"):
                if "sku" in line.lower() or "LV-" in line:
                    print(f"  → {line.strip()[:100]}")

        # Step 3: Extract cookies
        selenium_cookies = driver.get_cookies()
        print(f"[Session] Got {len(selenium_cookies)} cookies")

    finally:
        driver.quit()
        print("[Session] Chrome closed.")

    # Build requests.Session with the stolen cookies
    session = requests.Session()
    session.headers.update({
        "User-Agent": user_agent,
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br",
        "Referer": "https://www.luvansh.com/shop-diamond",
    })

    for c in selenium_cookies:
        session.cookies.set(
            c["name"], c["value"],
            domain=c.get("domain", ".luvansh.com"),
            path=c.get("path", "/"),
        )

    # Quick test with requests
    print("[Session] Testing cookies with requests ...")
    resp = session.get(test_url, timeout=15)
    print(f"[Session] Status: {resp.status_code}  Length: {len(resp.text)}")
    if resp.status_code == 200 and "SKU" in resp.text:
        print("[Session] Cookies work — requests can fetch product pages!")
    elif resp.status_code == 200:
        print("[Session] Got 200 but no SKU found — may need to check HTML structure")
    else:
        print(f"[Session] WARNING: Got status {resp.status_code} — cookies may not bypass Cloudflare")
        print("[Session] Falling back to Selenium for product pages ...")

    return session, resp.status_code == 200


# ── Run it ────────────────────────────────────────────────────
session, cookies_work = get_luvansh_session()
print(f"\nCookies bypass Cloudflare: {cookies_work}")

## 5. Step 2 — Fetch Product Pages (15 Threads via requests) → Get SKU + IGI

Uses `requests.Session` with the Cloudflare cookies grabbed above.
No Selenium needed — pure HTTP, true 15-thread parallelism.

**Fallback**: If cookies don't work, uses a single Selenium driver sequentially.

In [ ]:
print_lock = Lock()
progress_counter = {"done": 0}


def build_product_url(diamond):
    """Build the Luvansh product page URL from diamond data."""
    pid = diamond.get("product_id", "")
    carat = diamond.get("carat", 0)
    color = diamond.get("color", "").lower()
    clarity = diamond.get("clarity", "").lower()
    shape = diamond.get("shape", "round").lower()
    carat_str = str(carat).replace(".", "-")
    slug = f"igi-certified-{carat_str}-{color}-{clarity}-{shape}-lab-created-diamond-r"
    return f"https://www.luvansh.com/product/{pid}/{slug}"


def extract_sku_from_html(html_text):
    """Extract SKU from product page HTML."""
    for pat in [r'SKU\s*:\s*(LV-[A-Z0-9-]+)',
                r'SKU\s*:\s*([A-Z0-9][A-Z0-9-]+)',
                r'(LV-[A-Z]+-\d+)']:
        m = re.search(pat, html_text, re.I)
        if m:
            return m.group(1)
    return None


def sku_to_igi(sku):
    """Derive IGI report + PDF URL from SKU.
    LV-ROUNDEVVS2-726536327 → LG726536327
    """
    if not sku:
        return None, None
    parts = sku.split("-")
    if len(parts) >= 3:
        number = parts[-1]
        igi_report = f"LG{number}"
        pdf_url = f"https://api.igi.org/viewpdf.php?r={igi_report}"
        return igi_report, pdf_url
    return None, None


def fetch_product_page_requests(diamond, sess, total):
    """Fetch one product page via requests, extract SKU."""
    pid = diamond.get("product_id")
    if not pid or not diamond.get("color") or not diamond.get("clarity"):
        return {"sku": None, "igi_report": None, "igi_pdf_url": None, "error": "missing data"}

    url = build_product_url(diamond)
    result = {"sku": None, "igi_report": None, "igi_pdf_url": None, "error": None}

    try:
        resp = sess.get(url, timeout=15)
        if resp.status_code == 200:
            sku = extract_sku_from_html(resp.text)
            result["sku"] = sku
            result["igi_report"], result["igi_pdf_url"] = sku_to_igi(sku)
            if not sku:
                result["error"] = "SKU not in HTML"
        else:
            result["error"] = f"HTTP {resp.status_code}"
    except Exception as e:
        result["error"] = str(e)[:80]

    with print_lock:
        progress_counter["done"] += 1
        d = progress_counter["done"]
        status = result["sku"] or result.get("error", "?")
        if d % 50 == 0 or d <= 5 or d == total:
            print(f"  [{d}/{total}] pid={pid}  SKU={status}  IGI={result['igi_report'] or '-'}")

    return result


def fetch_product_page_selenium(diamond, driver, total):
    """Fallback: fetch one product page via Selenium."""
    pid = diamond.get("product_id")
    if not pid or not diamond.get("color") or not diamond.get("clarity"):
        return {"sku": None, "igi_report": None, "igi_pdf_url": None, "error": "missing data"}

    url = build_product_url(diamond)
    result = {"sku": None, "igi_report": None, "igi_pdf_url": None, "error": None}

    try:
        driver.get(url)
        time.sleep(3)
        page_text = driver.find_element(By.TAG_NAME, "body").text
        sku = extract_sku_from_html(page_text + " " + driver.page_source)
        result["sku"] = sku
        result["igi_report"], result["igi_pdf_url"] = sku_to_igi(sku)
        if not sku:
            result["error"] = "SKU not found"
    except Exception as e:
        result["error"] = str(e)[:80]

    with print_lock:
        progress_counter["done"] += 1
        d = progress_counter["done"]
        status = result["sku"] or result.get("error", "?")
        if d % 50 == 0 or d <= 5 or d == total:
            print(f"  [{d}/{total}] pid={pid}  SKU={status}  IGI={result['igi_report'] or '-'}")

    return result


# ── Run Step 2 ────────────────────────────────────────────────
total = len(all_diamonds)
progress_counter["done"] = 0

if cookies_work:
    # ── Fast path: 15 threads with requests ───────────────────
    print(f"Fetching {total} product pages with {THREADS} threads (requests) ...\n")
    with ThreadPoolExecutor(max_workers=THREADS) as executor:
        futures = {
            executor.submit(fetch_product_page_requests, d, session, total): i
            for i, d in enumerate(all_diamonds)
        }
        for future in as_completed(futures):
            idx = futures[future]
            result = future.result()
            all_diamonds[idx]["sku"] = result["sku"]
            all_diamonds[idx]["igi_report"] = result["igi_report"]
            all_diamonds[idx]["igi_pdf_url"] = result["igi_pdf_url"]
else:
    # ── Fallback: single Selenium driver (sequential) ─────────
    print(f"Cookies didn't work — using Selenium sequentially for {total} pages ...\n")
    driver = create_driver()
    try:
        for i, d in enumerate(all_diamonds):
            result = fetch_product_page_selenium(d, driver, total)
            all_diamonds[i]["sku"] = result["sku"]
            all_diamonds[i]["igi_report"] = result["igi_report"]
            all_diamonds[i]["igi_pdf_url"] = result["igi_pdf_url"]
            time.sleep(1)
    finally:
        driver.quit()

# ── Summary ───────────────────────────────────────────────────
found = sum(1 for d in all_diamonds if d.get("sku"))
print(f"\nDone! SKU found for {found}/{total} diamonds")

## 6. Step 3 — Download IGI PDFs (15 Threads) → Extract Proportions

In [ ]:
def extract_from_pdf(pdf_url):
    """
    Download an IGI PDF and extract proportions with pdfplumber.
    URL format: https://api.igi.org/viewpdf.php?r=LG726536327
    """
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/120.0.0.0 Safari/537.36"
    }
    resp = requests.get(pdf_url, headers=headers, timeout=30)
    resp.raise_for_status()

    props = {}
    with pdfplumber.open(io.BytesIO(resp.content)) as pdf:
        full_text = ""
        for page in pdf.pages:
            full_text += (page.extract_text() or "") + "\n"

    def find_f(pattern):
        m = re.search(pattern, full_text, re.I)
        return float(m.group(1)) if m else None

    def find_s(pattern):
        m = re.search(pattern, full_text, re.I)
        return m.group(1).strip() if m else None

    # ── Measurements & L/W ratio ──────────────────────────────
    meas_m = re.search(
        r'(\d+\.\d+\s*[-\u2013]\s*\d+\.\d+\s*[Xx\u00d7]\s*\d+\.\d+)', full_text)
    if meas_m:
        props["measurements"] = meas_m.group(1).strip()
        dims = re.findall(r'(\d+\.\d+)', meas_m.group(1))
        if len(dims) >= 2:
            l, w = float(dims[0]), float(dims[1])
            if min(l, w) > 0:
                props["lw_ratio"] = round(max(l, w) / min(l, w), 3)

    # ── Named fields ──────────────────────────────────────────
    props["table_pct"]      = find_f(r'Table\s*:?\s*(\d+(?:\.\d+)?)\s*%?')
    props["depth_pct"]      = find_f(r'Depth\s*:?\s*(\d+(?:\.\d+)?)\s*%?')
    props["crown_angle"]    = find_f(r'Crown\s*(?:Angle)?\s*:?\s*(\d+\.\d+)\s*[\u00b0]?')
    props["pavilion_angle"] = find_f(r'Pavilion\s*(?:Angle)?\s*:?\s*(\d+\.\d+)\s*[\u00b0]?')
    props["crown_height"]   = find_f(r'Crown\s*(?:Height)?\s*:?\s*(\d+\.\d+)\s*%?')
    props["pavilion_depth"] = find_f(r'Pavilion\s*(?:Depth)?\s*:?\s*(\d+\.\d+)\s*%?')

    # ── Proportions diagram: "13.5% 58% 33.1° 40.9° 43% Pointed 61%" ──
    prop_m = re.search(
        r'(\d+\.\d+)%\s+(\d+)%\s+(\d+\.\d+)[\u00b0]\s+(\d+\.\d+)[\u00b0]\s+'
        r'(\d+(?:\.\d+)?)%\s+\w+\s+(\d+(?:\.\d+)?)%',
        full_text)
    if prop_m:
        props.setdefault("crown_height",   float(prop_m.group(1)))
        props.setdefault("table_pct",      float(prop_m.group(2)))
        props.setdefault("crown_angle",    float(prop_m.group(3)))
        props.setdefault("pavilion_angle", float(prop_m.group(4)))
        props.setdefault("pavilion_depth", float(prop_m.group(5)))
        props.setdefault("depth_pct",      float(prop_m.group(6)))

    # ── Two angles side by side ───────────────────────────────
    if not props.get("crown_angle") or not props.get("pavilion_angle"):
        ang_m = re.search(r'(\d{2}\.\d+)[\u00b0]\s+(\d{2}\.\d+)[\u00b0]', full_text)
        if ang_m:
            props.setdefault("crown_angle",   float(ang_m.group(1)))
            props.setdefault("pavilion_angle", float(ang_m.group(2)))

    # ── Grading fields ────────────────────────────────────────
    props["polish"]       = find_s(r'Polish\s*:?\s*(EXCELLENT|VERY\s*GOOD|GOOD|FAIR|POOR)')
    props["symmetry"]     = find_s(r'Symmetry\s*:?\s*(EXCELLENT|VERY\s*GOOD|GOOD|FAIR|POOR)')
    props["fluorescence"] = find_s(r'Fluorescence\s*:?\s*(NONE|FAINT|MEDIUM|STRONG|VERY\s*STRONG)')
    props["girdle"]       = find_s(r'Girdle\s*:?\s*([A-Za-z\s]+(?:\(Faceted\))?)')
    props["culet"]        = find_s(r'Culet\s*:?\s*(None|Pointed|Very\s*Small|Small|Medium|Large)')
    props["cut_grade_igi"] = find_s(r'Cut\s*(?:Grade)?\s*:?\s*(IDEAL|EXCELLENT|VERY\s*GOOD|GOOD)')
    props["carat_igi"]    = find_f(r'Carat\s*Weight\s*:?\s*(\d+\.\d+)')
    props["color_igi"]    = find_s(r'Color\s*Grade\s*:?\s*([A-Z])')
    props["clarity_igi"]  = find_s(r'Clarity\s*Grade\s*:?\s*(\w+\s*\d*)')

    return props


pdf_progress = {"done": 0, "total": 0}
pdf_progress_lock = Lock()


def fetch_igi_data(diamond):
    """
    Download the IGI PDF for one diamond and extract proportions.
    Uses requests only (no Selenium) — safe for threading.
    """
    pdf_url = diamond.get("igi_pdf_url")
    igi_report = diamond.get("igi_report", "")
    props = {}

    if not pdf_url:
        props["pdf_error"] = "no URL"
    else:
        try:
            props = extract_from_pdf(pdf_url)
        except Exception as e:
            props["pdf_error"] = str(e)[:120]

    with pdf_progress_lock:
        pdf_progress["done"] += 1
        d, t = pdf_progress["done"], pdf_progress["total"]

    with print_lock:
        tbl = props.get('table_pct', '-')
        dep = props.get('depth_pct', '-')
        err = props.get('pdf_error', '')
        status = f"Table={tbl}% Depth={dep}%" if not err else f"ERROR: {err}"
        print(f"  [{d}/{t}] {igi_report:<14} {status}")

    return props


# ── Run threaded PDF downloads ────────────────────────────────
diamonds_with_pdf = [d for d in all_diamonds if d.get("igi_pdf_url")]
print(f"Downloading {len(diamonds_with_pdf)} IGI PDFs with {THREADS} threads ...\n")

pdf_progress["done"] = 0
pdf_progress["total"] = len(diamonds_with_pdf)

with ThreadPoolExecutor(max_workers=THREADS) as executor:
    futures = {
        executor.submit(fetch_igi_data, d): i
        for i, d in enumerate(diamonds_with_pdf)
    }
    for future in as_completed(futures):
        idx = futures[future]
        props = future.result()
        # Store all IGI props back into the diamond dict
        for k, v in props.items():
            diamonds_with_pdf[idx][f"igi_{k}"] = v

# Also tag diamonds that had no PDF URL
for d in all_diamonds:
    if not d.get("igi_pdf_url"):
        d["igi_pdf_error"] = "no SKU/URL"

ok = sum(1 for d in all_diamonds if d.get("igi_table_pct") is not None)
print(f"\nIGI data extracted for {ok}/{len(all_diamonds)} diamonds")

## 7. Step 4 — Merge All Data → `diamonds_full.csv`

In [ ]:
# ── Build final DataFrame with ALL columns ────────────────────
rows = []
for d in all_diamonds:
    rows.append({
        # ── Original data from HTML ──
        "product_id":       d.get("product_id"),
        "shape":            d.get("shape"),
        "carat":            d.get("carat"),
        "color":            d.get("color"),
        "clarity":          d.get("clarity"),
        "cut":              d.get("cut"),
        "price_original":   d.get("price_original"),
        "price_discounted": d.get("price_discounted"),
        # ── From product page ──
        "sku":              d.get("sku"),
        "igi_report":       d.get("igi_report"),
        "igi_pdf_url":      d.get("igi_pdf_url"),
        # ── From IGI PDF ──
        "igi_measurements": d.get("igi_measurements"),
        "igi_lw_ratio":     d.get("igi_lw_ratio"),
        "igi_table_pct":    d.get("igi_table_pct"),
        "igi_depth_pct":    d.get("igi_depth_pct"),
        "igi_crown_angle":  d.get("igi_crown_angle"),
        "igi_pavilion_angle": d.get("igi_pavilion_angle"),
        "igi_crown_height": d.get("igi_crown_height"),
        "igi_pavilion_depth": d.get("igi_pavilion_depth"),
        "igi_polish":       d.get("igi_polish"),
        "igi_symmetry":     d.get("igi_symmetry"),
        "igi_fluorescence": d.get("igi_fluorescence"),
        "igi_girdle":       d.get("igi_girdle"),
        "igi_culet":        d.get("igi_culet"),
        "igi_cut_grade":    d.get("igi_cut_grade_igi"),
        "igi_carat":        d.get("igi_carat_igi"),
        "igi_color":        d.get("igi_color_igi"),
        "igi_clarity":      d.get("igi_clarity_igi"),
        "igi_pdf_error":    d.get("igi_pdf_error"),
    })

df_full = pd.DataFrame(rows)
df_full.to_csv(FULL_CSV, index=False)
print(f"Saved → {FULL_CSV}  ({len(df_full)} rows, {len(df_full.columns)} columns)")

# ── Summary stats ─────────────────────────────────────────────
has_sku = df_full["sku"].notna().sum()
has_igi = df_full["igi_table_pct"].notna().sum()
print(f"\nSKU extracted:     {has_sku}/{len(df_full)}")
print(f"IGI data extracted: {has_igi}/{len(df_full)}")

print(f"\nColumns: {list(df_full.columns)}")
df_full.head(10)

In [ ]:
# ── Download CSVs in Colab ────────────────────────────────────
try:
    from google.colab import files
    files.download(RAW_CSV)
    files.download(FULL_CSV)
except ImportError:
    print(f"Files saved locally: {RAW_CSV}, {FULL_CSV}")